# Clasificación del riesgo de diabetes mediante aprendizaje automático

## Comparación de estrategias de preprocesamiento y modelos

**Autor:** Alexander Marín  
**Área:** Analítica de datos aplicada al sector salud

### Objetivo

Comparar el desempeño de diferentes modelos de clasificación y estrategias para
el tratamiento de los datos faltantes encubiertos del conjunto Pima Indians
Diabetes.

### Estrategias de datos faltantes

1. Análisis de casos completos.
2. Imputación mediante la mediana.
3. Imputación mediante la mediana con indicadores de ausencia.

### Modelos

- Clasificador dummy como referencia mínima.
- Regresión logística como modelo base interpretable.
- Random Forest.
- XGBoost.

### Evaluación

La comparación utilizará validación cruzada estratificada sobre los datos de
entrenamiento. El conjunto de prueba permanecerá reservado hasta seleccionar la
estrategia final.

Las métricas principales serán sensibilidad, especificidad, precisión, F1,
ROC-AUC y PR-AUC. La exactitud se reportará como medida complementaria.

Este análisis tiene fines educativos y no constituye una herramienta
diagnóstica ni una validación clínica.

In [1]:
# importar librerias
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold
)

pd.set_option(
    "display.max_columns",
    None
)

sns.set_theme(
    style="whitegrid",
    palette="deep"
)

RANDOM_STATE = 42
TEST_SIZE = 0.20
N_SPLITS = 5

In [2]:
# cargar datos
DATA_PATH = "/content/1_diabetes.csv"

df = pd.read_csv(DATA_PATH)

print(f"Registros: {df.shape[0]}")
print(f"Variables: {df.shape[1]}")

df.head()

Registros: 768
Variables: 9


,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


In [3]:
invalid_zero_columns = [
    "Glucose",
    "BloodPressure",
    "SkinThickness",
    "Insulin",
    "BMI"
]

X = (
    df
    .drop(columns="Outcome")
    .copy()
)

y = (
    df["Outcome"]
    .copy()
)

# Conversión determinística de ceros implausibles en NaN
X[invalid_zero_columns] = (
    X[invalid_zero_columns]
    .replace(0, np.nan)
)

print(f"Dimensiones de X: {X.shape}")
print(f"Dimensiones de y: {y.shape}")

print("\nDatos faltantes en X:")
print(X.isna().sum())

Dimensiones de X: (768, 8)
Dimensiones de y: (768,)

Datos faltantes en X:
Pregnancies                   0
Glucose                       5
BloodPressure                35
SkinThickness               227
Insulin                     374
BMI                          11
DiabetesPedigreeFunction      0
Age                           0
dtype: int64


In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y
)

print(f"Registros de entrenamiento: {len(X_train)}")
print(f"Registros de prueba: {len(X_test)}")

print(
    "\nProporción positiva en el dataset completo:",
    round(y.mean(), 3)
)

print(
    "Proporción positiva en entrenamiento:",
    round(y_train.mean(), 3)
)

print(
    "Proporción positiva en prueba:",
    round(y_test.mean(), 3)
)

Registros de entrenamiento: 614
Registros de prueba: 154

Proporción positiva en el dataset completo: 0.349
Proporción positiva en entrenamiento: 0.349
Proporción positiva en prueba: 0.351


In [5]:
partition_summary = pd.DataFrame({
    "particion": [
        "Dataset completo",
        "Entrenamiento",
        "Prueba"
    ],
    "n_registros": [
        len(y),
        len(y_train),
        len(y_test)
    ],
    "n_negativos": [
        y.eq(0).sum(),
        y_train.eq(0).sum(),
        y_test.eq(0).sum()
    ],
    "n_positivos": [
        y.eq(1).sum(),
        y_train.eq(1).sum(),
        y_test.eq(1).sum()
    ],
    "porcentaje_positivo": [
        y.mean() * 100,
        y_train.mean() * 100,
        y_test.mean() * 100
    ]
})

partition_summary.round(2)

,particion,n_registros,n_negativos,n_positivos,porcentaje_positivo
0,Dataset completo,768,500,268,34.90
1,Entrenamiento,614,400,214,34.85
2,Prueba,154,100,54,35.06


## Separación entre entrenamiento y prueba

El 80 % de los registros se asignó al conjunto de entrenamiento y el 20 % al
conjunto de prueba.

La separación se realizó mediante estratificación de `Outcome`, conservando una
proporción similar de resultados positivos y negativos en ambas particiones.

El conjunto de prueba permanecerá reservado y no será utilizado para:

- Calcular medianas de imputación.
- Seleccionar modelos.
- Ajustar hiperparámetros.
- Elegir umbrales de clasificación.
- Comparar estrategias durante el desarrollo.

Las decisiones se tomarán mediante validación cruzada estratificada dentro del
conjunto de entrenamiento. El conjunto de prueba se utilizará una sola vez para
la evaluación final.

## Distribución de los datos faltantes por partición

Antes de construir los pipelines, se verifica la cantidad de datos faltantes en
entrenamiento y prueba.

Esta revisión no modifica los datos. Su propósito es documentar que ambas
particiones conservan el problema de ausencia identificado durante el análisis
exploratorio.

In [6]:
missing_by_partition = pd.concat(
    {
        "Entrenamiento": X_train.isna().sum(),
        "Prueba": X_test.isna().sum()
    },
    axis=1
)

missing_by_partition["Total"] = (
    missing_by_partition["Entrenamiento"]
    + missing_by_partition["Prueba"]
)

missing_by_partition

,Entrenamiento,Prueba,Total
Pregnancies,0,0,0
Glucose,4,1,5
BloodPressure,23,12,35
SkinThickness,175,52,227
Insulin,290,84,374
BMI,9,2,11
DiabetesPedigreeFunction,0,0,0
Age,0,0,0


## Estrategias de tratamiento de datos faltantes

Se compararán tres estrategias utilizando inicialmente el mismo modelo de
regresión logística:

1. **Casos completos:** se utilizan únicamente registros sin datos faltantes.
2. **Imputación por mediana:** los faltantes se reemplazan con medianas aprendidas
   dentro de cada pliegue de entrenamiento.
3. **Mediana con indicadores:** además de imputar, se agregan variables binarias
   que indican qué mediciones estaban ausentes.

El escenario de casos completos representa una población diferente y utiliza
menos registros. Por tanto, sus métricas se presentan como análisis de
sensibilidad y no son directamente equivalentes a las obtenidas con toda la
muestra.

In [7]:
complete_case_mask_train = (
    X_train
    .notna()
    .all(axis=1)
)

X_train_complete = (
    X_train
    .loc[complete_case_mask_train]
    .copy()
)

y_train_complete = (
    y_train
    .loc[complete_case_mask_train]
    .copy()
)

complete_case_summary = pd.DataFrame({
    "escenario": [
        "Entrenamiento completo",
        "Casos completos"
    ],
    "n_registros": [
        len(X_train),
        len(X_train_complete)
    ],
    "n_negativos": [
        y_train.eq(0).sum(),
        y_train_complete.eq(0).sum()
    ],
    "n_positivos": [
        y_train.eq(1).sum(),
        y_train_complete.eq(1).sum()
    ],
    "porcentaje_positivo": [
        y_train.mean() * 100,
        y_train_complete.mean() * 100
    ]
})

complete_case_summary.round(2)

,escenario,n_registros,n_negativos,n_positivos,porcentaje_positivo
0,Entrenamiento completo,614,400,214,34.85
1,Casos completos,322,213,109,33.85


In [8]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression

from sklearn.model_selection import cross_validate

from sklearn.metrics import (
    make_scorer,
    precision_score,
    recall_score,
    f1_score
)

## Validación cruzada y métricas

Se utiliza validación cruzada estratificada de cinco pliegues. En cada iteración,
cuatro pliegues se utilizan para entrenar y el restante para validar.

La imputación y el escalado forman parte de los pipelines. Por tanto, sus
parámetros se aprenden nuevamente dentro de cada pliegue, evitando fuga de
información.

Se reportan:

- Exactitud.
- Precisión.
- Sensibilidad.
- Especificidad.
- F1.
- ROC-AUC.
- PR-AUC.

In [9]:
cross_validation = StratifiedKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=RANDOM_STATE
)

scoring_metrics = {
    "accuracy": "accuracy",

    "precision": make_scorer(
        precision_score,
        zero_division=0
    ),

    "sensitivity": make_scorer(
        recall_score,
        pos_label=1,
        zero_division=0
    ),

    "specificity": make_scorer(
        recall_score,
        pos_label=0,
        zero_division=0
    ),

    "f1": make_scorer(
        f1_score,
        zero_division=0
    ),

    "roc_auc": "roc_auc",

    "pr_auc": "average_precision"
}

In [10]:
logistic_model = LogisticRegression(
    max_iter=2000,
    random_state=RANDOM_STATE
)

complete_case_pipeline = Pipeline(
    steps=[
        (
            "scaler",
            StandardScaler()
        ),
        (
            "model",
            logistic_model
        )
    ]
)

median_imputation_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="median",
                add_indicator=False
            )
        ),
        (
            "scaler",
            StandardScaler()
        ),
        (
            "model",
            LogisticRegression(
                max_iter=2000,
                random_state=RANDOM_STATE
            )
        )
    ]
)

median_indicator_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="median",
                add_indicator=True
            )
        ),
        (
            "scaler",
            StandardScaler()
        ),
        (
            "model",
            LogisticRegression(
                max_iter=2000,
                random_state=RANDOM_STATE
            )
        )
    ]
)

In [11]:
strategy_configurations = {
    "Casos completos": {
        "pipeline": complete_case_pipeline,
        "X": X_train_complete,
        "y": y_train_complete
    },

    "Imputación por mediana": {
        "pipeline": median_imputation_pipeline,
        "X": X_train,
        "y": y_train
    },

    "Mediana + indicadores": {
        "pipeline": median_indicator_pipeline,
        "X": X_train,
        "y": y_train
    }
}

strategy_results = []

for strategy_name, configuration in strategy_configurations.items():

    cv_results = cross_validate(
        estimator=configuration["pipeline"],
        X=configuration["X"],
        y=configuration["y"],
        cv=cross_validation,
        scoring=scoring_metrics,
        n_jobs=-1,
        return_train_score=False
    )

    result_row = {
        "estrategia": strategy_name,
        "n_registros": len(configuration["X"])
    }

    for metric_name in scoring_metrics:

        metric_values = cv_results[
            f"test_{metric_name}"
        ]

        result_row[f"{metric_name}_media"] = (
            metric_values.mean()
        )

        result_row[f"{metric_name}_sd"] = (
            metric_values.std()
        )

    strategy_results.append(result_row)

strategy_results = (
    pd.DataFrame(strategy_results)
    .set_index("estrategia")
    .round(3)
)

strategy_results

,n_registros,accuracy_media,accuracy_sd,precision_media,precision_sd,sensitivity_media,sensitivity_sd,specificity_media,specificity_sd,f1_media,f1_sd,roc_auc_media,roc_auc_sd,pr_auc_media,pr_auc_sd
estrategia,,,,,,,,,,,,,,,
Casos completos,322,0.801,0.029,0.768,0.080,0.615,0.082,0.897,0.058,0.675,0.050,0.858,0.033,0.760,0.055
Imputación por mediana,614,0.788,0.019,0.764,0.060,0.575,0.023,0.903,0.031,0.655,0.024,0.843,0.019,0.747,0.058
Mediana + indicadores,614,0.790,0.021,0.753,0.057,0.598,0.034,0.893,0.032,0.665,0.029,0.839,0.023,0.746,0.055


### Interpretación de las estrategias

El análisis de casos completos obtuvo las métricas medias más altas, con un
ROC-AUC de 0,858 y una sensibilidad de 0,615. Sin embargo, esta estrategia
utilizó solamente 322 de los 614 registros de entrenamiento.

La eliminación de registros modifica la población analizada y puede favorecer
artificialmente el desempeño si los casos completos presentan características
diferentes. Por esta razón, sus resultados se consideran un análisis de
sensibilidad y no una comparación directamente equivalente.

Entre las estrategias evaluadas sobre los mismos 614 registros, la incorporación
de indicadores de ausencia aumentó la sensibilidad de 0,575 a 0,598 y el F1 de
0,655 a 0,665 frente a la imputación por mediana sin indicadores.

Las diferencias en ROC-AUC y PR-AUC fueron pequeñas respecto a la variabilidad
entre pliegues. En consecuencia, se selecciona la estrategia de mediana con
indicadores porque:

- Conserva todos los registros de entrenamiento.
- Evita el sesgo potencial de eliminar casos incompletos.
- Mejora la identificación de resultados positivos.
- Permite representar la información contenida en la ausencia de mediciones.
- Es coherente con los patrones de ausencia encontrados durante la exploración.

La mediana y los indicadores serán aprendidos dentro de cada pliegue mediante
pipelines.

## Comparación inicial de modelos

Se comparan cuatro clasificadores utilizando la misma validación cruzada y, salvo
el modelo dummy, la estrategia de imputación por mediana con indicadores de
ausencia.

El clasificador dummy establece una referencia mínima basada únicamente en la
distribución de las clases. Un modelo útil debe superar claramente esta
referencia.

La regresión logística funciona como modelo base interpretable. Random Forest y
XGBoost permiten representar relaciones no lineales e interacciones entre
variables.

En esta etapa se utilizan configuraciones iniciales. El ajuste de hiperparámetros
se realizará posteriormente y solamente sobre el conjunto de entrenamiento.

In [12]:
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

In [13]:
dummy_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="median",
                add_indicator=True
            )
        ),
        (
            "model",
            DummyClassifier(
                strategy="prior"
            )
        )
    ]
)

logistic_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="median",
                add_indicator=True
            )
        ),
        (
            "scaler",
            StandardScaler()
        ),
        (
            "model",
            LogisticRegression(
                max_iter=2000,
                random_state=RANDOM_STATE
            )
        )
    ]
)

random_forest_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="median",
                add_indicator=True
            )
        ),
        (
            "model",
            RandomForestClassifier(
                n_estimators=500,
                random_state=RANDOM_STATE,
                n_jobs=1
            )
        )
    ]
)

xgboost_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="median",
                add_indicator=True
            )
        ),
        (
            "model",
            XGBClassifier(
                n_estimators=300,
                learning_rate=0.05,
                max_depth=3,
                subsample=0.80,
                colsample_bytree=0.80,
                objective="binary:logistic",
                eval_metric="logloss",
                random_state=RANDOM_STATE,
                n_jobs=1
            )
        )
    ]
)

model_pipelines = {
    "Dummy": dummy_pipeline,
    "Regresión logística": logistic_pipeline,
    "Random Forest": random_forest_pipeline,
    "XGBoost": xgboost_pipeline
}

In [14]:
model_results = []
model_fold_results = {}

for model_name, model_pipeline in model_pipelines.items():

    cv_results = cross_validate(
        estimator=model_pipeline,
        X=X_train,
        y=y_train,
        cv=cross_validation,
        scoring=scoring_metrics,
        n_jobs=-1,
        return_train_score=False
    )

    model_fold_results[model_name] = cv_results

    result_row = {
        "modelo": model_name,
        "n_registros": len(X_train)
    }

    for metric_name in scoring_metrics:

        metric_values = cv_results[
            f"test_{metric_name}"
        ]

        result_row[f"{metric_name}_media"] = (
            metric_values.mean()
        )

        result_row[f"{metric_name}_sd"] = (
            metric_values.std()
        )

    model_results.append(result_row)

model_results = (
    pd.DataFrame(model_results)
    .set_index("modelo")
    .sort_values(
        "roc_auc_media",
        ascending=False
    )
    .round(3)
)

model_results

,n_registros,accuracy_media,accuracy_sd,precision_media,precision_sd,sensitivity_media,sensitivity_sd,specificity_media,specificity_sd,f1_media,f1_sd,roc_auc_media,roc_auc_sd,pr_auc_media,pr_auc_sd
modelo,,,,,,,,,,,,,,,
Regresión logística,614,0.790,0.021,0.753,0.057,0.598,0.034,0.893,0.032,0.665,0.029,0.839,0.023,0.746,0.055
Random Forest,614,0.761,0.021,0.690,0.051,0.580,0.036,0.857,0.037,0.628,0.023,0.820,0.023,0.710,0.039
XGBoost,614,0.757,0.034,0.679,0.078,0.598,0.033,0.842,0.055,0.633,0.036,0.812,0.028,0.697,0.058
Dummy,614,0.651,0.002,0.000,0.000,0.000,0.000,1.000,0.000,0.000,0.000,0.500,0.000,0.349,0.002


### Interpretación de la comparación inicial

Todos los modelos entrenados superaron claramente al clasificador dummy en
ROC-AUC y PR-AUC.

El clasificador dummy obtuvo una exactitud de 0,651 porque asignó todos los
registros a la clase negativa. Como consecuencia, alcanzó especificidad de 1,00,
pero sensibilidad de 0,00. Este resultado demuestra por qué la exactitud no debe
interpretarse de forma aislada.

La regresión logística presentó el mejor desempeño inicial:

- ROC-AUC: 0,839.
- PR-AUC: 0,746.
- F1: 0,665.
- Sensibilidad: 0,598.
- Especificidad: 0,893.

Random Forest y XGBoost no superaron a la regresión logística con las
configuraciones evaluadas. XGBoost igualó su sensibilidad media, pero presentó
menor precisión, especificidad, F1, ROC-AUC y PR-AUC.

Los resultados muestran que una mayor complejidad algorítmica no garantiza un
mejor desempeño. En este dataset, la regresión logística ofrece inicialmente el
mejor equilibrio entre discriminación, estabilidad e interpretabilidad.

Estas conclusiones corresponden a configuraciones iniciales. Antes de seleccionar
el modelo final se realizará ajuste de hiperparámetros exclusivamente sobre los
datos de entrenamiento.

## Ajuste de hiperparámetros de la regresión logística

La regresión logística presentó el mejor desempeño inicial y será ajustada
mediante búsqueda en cuadrícula.

Se evaluarán diferentes niveles de regularización, penalizaciones L1 y L2, y el
uso de pesos balanceados.

La búsqueda utilizará exclusivamente los datos de entrenamiento y validación
cruzada estratificada. El criterio principal de selección será ROC-AUC, debido a
que evalúa la capacidad de ordenar positivos y negativos sin fijar todavía un
umbral de clasificación.

In [15]:
from sklearn.model_selection import GridSearchCV

In [16]:
tuned_logistic_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="median",
                add_indicator=True
            )
        ),
        (
            "scaler",
            StandardScaler()
        ),
        (
            "model",
            LogisticRegression(
                solver="liblinear",
                max_iter=3000,
                random_state=RANDOM_STATE
            )
        )
    ]
)

logistic_parameter_grid = {
    "model__C": np.logspace(
        -3,
        2,
        11
    ),

    "model__penalty": [
        "l1",
        "l2"
    ],

    "model__class_weight": [
        None,
        "balanced"
    ]
}

logistic_grid_search = GridSearchCV(
    estimator=tuned_logistic_pipeline,
    param_grid=logistic_parameter_grid,
    scoring=scoring_metrics,
    refit="roc_auc",
    cv=cross_validation,
    n_jobs=-1,
    return_train_score=True
)

logistic_grid_search.fit(
    X_train,
    y_train
)

print("Mejores parámetros:")

for parameter, value in logistic_grid_search.best_params_.items():
    print(f"{parameter}: {value}")

print(
    "\nMejor ROC-AUC medio:",
    round(
        logistic_grid_search.best_score_,
        3
    )
)

Mejores parámetros:
model__C: 0.1
model__class_weight: balanced
model__penalty: l1

Mejor ROC-AUC medio: 0.844


In [17]:
logistic_search_results = pd.DataFrame(
    logistic_grid_search.cv_results_
)

logistic_top_results = (
    logistic_search_results[
        [
            "param_model__C",
            "param_model__penalty",
            "param_model__class_weight",
            "mean_test_accuracy",
            "mean_test_precision",
            "mean_test_sensitivity",
            "mean_test_specificity",
            "mean_test_f1",
            "mean_test_roc_auc",
            "std_test_roc_auc",
            "mean_test_pr_auc"
        ]
    ]
    .sort_values(
        "mean_test_roc_auc",
        ascending=False
    )
    .head(10)
    .reset_index(drop=True)
    .round(3)
)

logistic_top_results

,param_model__C,param_model__penalty,param_model__class_weight,mean_test_accuracy,mean_test_precision,mean_test_sensitivity,mean_test_specificity,mean_test_f1,mean_test_roc_auc,std_test_roc_auc,mean_test_pr_auc
0,0.100,l1,balanced,0.749,0.618,0.738,0.755,0.672,0.844,0.019,0.751
1,0.100,l1,None,0.788,0.750,0.593,0.892,0.662,0.843,0.022,0.752
2,0.032,l2,balanced,0.746,0.618,0.720,0.760,0.664,0.842,0.017,0.740
3,0.032,l2,None,0.782,0.722,0.612,0.872,0.661,0.842,0.016,0.742
4,0.316,l1,balanced,0.761,0.640,0.734,0.775,0.681,0.841,0.020,0.747
5,0.100,l2,None,0.787,0.740,0.603,0.885,0.663,0.841,0.020,0.746
6,0.100,l2,balanced,0.759,0.637,0.724,0.778,0.677,0.841,0.020,0.745
7,0.316,l1,None,0.782,0.739,0.584,0.888,0.651,0.840,0.020,0.747
8,0.010,l2,None,0.777,0.703,0.626,0.857,0.661,0.840,0.014,0.731
9,0.010,l2,balanced,0.739,0.603,0.743,0.738,0.665,0.840,0.013,0.730


### Interpretación del ajuste de la regresión logística

La configuración seleccionada utilizó regularización L1, `C = 0,1` y pesos de
clase balanceados.

El ROC-AUC aumentó de 0,839 en la configuración inicial a 0,844 después del
ajuste. La mejora absoluta fue de 0,005, por lo que el ajuste no produjo un cambio
sustancial en la capacidad global de discriminación.

El uso de pesos balanceados modificó de manera importante el comportamiento en
el umbral predeterminado:

- La sensibilidad aumentó de 0,598 a 0,738.
- La especificidad disminuyó de 0,893 a 0,755.
- La precisión disminuyó de 0,753 a 0,618.
- El F1 aumentó ligeramente de 0,665 a 0,672.

El modelo balanceado identifica una mayor proporción de resultados positivos,
pero a costa de clasificar más registros negativos como positivos.

Este intercambio puede resultar relevante en un escenario de tamizaje, donde
los falsos negativos podrían tener un costo mayor. Sin embargo, el proyecto no
establece que dicho equilibrio sea clínicamente óptimo.

La diferencia mínima entre las mejores configuraciones también muestra que la
selección del modelo no debe basarse únicamente en el mayor valor observado en
validación cruzada. Posteriormente se evaluará el umbral de clasificación y la
calibración de las probabilidades.

## Ajuste de hiperparámetros de Random Forest

Random Forest se ajustará mediante búsqueda aleatoria para explorar diferentes
niveles de complejidad sin evaluar todas las combinaciones posibles.

Se modificarán:

- Número de árboles.
- Profundidad máxima.
- Número mínimo de registros para dividir un nodo.
- Número mínimo de registros por hoja.
- Cantidad de variables consideradas en cada división.
- Uso de pesos de clase.

El criterio de selección será ROC-AUC y el conjunto de prueba permanecerá
reservado.

In [18]:
from sklearn.model_selection import RandomizedSearchCV

In [19]:
random_forest_search_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="median",
                add_indicator=True
            )
        ),
        (
            "model",
            RandomForestClassifier(
                random_state=RANDOM_STATE,
                n_jobs=1
            )
        )
    ]
)

random_forest_parameter_space = {
    "model__n_estimators": [
        300,
        500,
        800
    ],

    "model__max_depth": [
        None,
        3,
        5,
        8,
        12
    ],

    "model__min_samples_split": [
        2,
        5,
        10,
        20
    ],

    "model__min_samples_leaf": [
        1,
        2,
        4,
        8
    ],

    "model__max_features": [
        "sqrt",
        "log2",
        0.5,
        None
    ],

    "model__class_weight": [
        None,
        "balanced",
        "balanced_subsample"
    ]
}

random_forest_search = RandomizedSearchCV(
    estimator=random_forest_search_pipeline,
    param_distributions=random_forest_parameter_space,
    n_iter=40,
    scoring=scoring_metrics,
    refit="roc_auc",
    cv=cross_validation,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    return_train_score=True
)

random_forest_search.fit(
    X_train,
    y_train
)

print("Mejores parámetros:")

for parameter, value in random_forest_search.best_params_.items():
    print(f"{parameter}: {value}")

print(
    "\nMejor ROC-AUC medio:",
    round(
        random_forest_search.best_score_,
        3
    )
)

Mejores parámetros:
model__n_estimators: 500
model__min_samples_split: 2
model__min_samples_leaf: 2
model__max_features: 0.5
model__max_depth: 5
model__class_weight: balanced

Mejor ROC-AUC medio: 0.84


In [20]:
random_forest_search_results = pd.DataFrame(
    random_forest_search.cv_results_
)

random_forest_top_results = (
    random_forest_search_results[
        [
            "param_model__n_estimators",
            "param_model__max_depth",
            "param_model__min_samples_split",
            "param_model__min_samples_leaf",
            "param_model__max_features",
            "param_model__class_weight",
            "mean_test_accuracy",
            "mean_test_precision",
            "mean_test_sensitivity",
            "mean_test_specificity",
            "mean_test_f1",
            "mean_test_roc_auc",
            "std_test_roc_auc",
            "mean_test_pr_auc"
        ]
    ]
    .sort_values(
        "mean_test_roc_auc",
        ascending=False
    )
    .head(10)
    .reset_index(drop=True)
    .round(3)
)

random_forest_top_results

,param_model__n_estimators,param_model__max_depth,param_model__min_samples_split,param_model__min_samples_leaf,param_model__max_features,param_model__class_weight,mean_test_accuracy,mean_test_precision,mean_test_sensitivity,mean_test_specificity,mean_test_f1,mean_test_roc_auc,std_test_roc_auc,mean_test_pr_auc
0,500,5,2,2,0.5,balanced,0.765,0.649,0.719,0.790,0.682,0.840,0.021,0.736
1,300,5,5,8,log2,None,0.774,0.746,0.542,0.898,0.625,0.838,0.017,0.734
2,800,3,10,8,0.5,balanced_subsample,0.774,0.655,0.743,0.790,0.696,0.837,0.019,0.739
3,500,3,10,1,0.5,balanced,0.767,0.650,0.724,0.790,0.684,0.837,0.017,0.739
4,800,12,5,8,log2,None,0.772,0.723,0.570,0.880,0.634,0.837,0.018,0.728
5,800,None,2,8,0.5,balanced,0.774,0.660,0.729,0.798,0.692,0.837,0.021,0.728
6,300,3,20,2,0.5,balanced,0.767,0.648,0.729,0.788,0.686,0.836,0.019,0.738
7,800,5,20,2,log2,balanced_subsample,0.764,0.642,0.738,0.778,0.685,0.836,0.022,0.726
8,500,5,2,8,log2,balanced,0.764,0.641,0.743,0.775,0.687,0.836,0.022,0.724
9,500,5,2,4,log2,balanced_subsample,0.774,0.652,0.757,0.782,0.700,0.836,0.021,0.729


### Interpretación del ajuste de Random Forest

El ajuste aumentó el ROC-AUC medio de Random Forest de 0,820 a 0,840, una mejora
absoluta de 0,020.

La mejor configuración utilizó:

- 500 árboles.
- Profundidad máxima de 5.
- Mínimo de 2 registros por hoja.
- 50 % de las variables disponibles en cada división.
- Pesos de clase balanceados.

La profundidad limitada y el mínimo de registros por hoja reducen la complejidad
de los árboles individuales y ayudan a controlar el sobreajuste.

El modelo ajustado alcanzó:

- Sensibilidad: 0,719.
- Especificidad: 0,790.
- Precisión: 0,649.
- F1: 0,682.
- ROC-AUC: 0,840.
- PR-AUC: 0,736.

Random Forest quedó muy cerca de la regresión logística ajustada. Presentó mayor
especificidad, precisión y F1, mientras que la regresión logística conservó una
pequeña ventaja en sensibilidad, ROC-AUC y PR-AUC.

Las diferencias se encuentran dentro de rangos cercanos a la variabilidad entre
pliegues, por lo que todavía no puede afirmarse que uno de los dos modelos sea
claramente superior.

## Ajuste de hiperparámetros de XGBoost

XGBoost se ajustará mediante búsqueda aleatoria sobre 25 configuraciones.

Se evaluarán parámetros relacionados con:

- Número y profundidad de los árboles.
- Velocidad de aprendizaje.
- Submuestreo de registros y variables.
- Complejidad mínima requerida para crear divisiones.
- Regularización.
- Peso asignado a la clase positiva.

Se utiliza el método de construcción `hist` para reducir el tiempo de
entrenamiento. La selección se realiza mediante ROC-AUC sobre validación cruzada
y sin utilizar el conjunto de prueba.

In [21]:
positive_class_weight = (
    y_train.eq(0).sum()
    / y_train.eq(1).sum()
)

round(
    positive_class_weight,
    3
)

np.float64(1.869)

In [22]:
xgboost_search_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="median",
                add_indicator=True
            )
        ),
        (
            "model",
            XGBClassifier(
                objective="binary:logistic",
                eval_metric="logloss",
                tree_method="hist",
                random_state=RANDOM_STATE,
                n_jobs=1
            )
        )
    ]
)

xgboost_parameter_space = {
    "model__n_estimators": [
        100,
        200,
        300
    ],

    "model__learning_rate": [
        0.02,
        0.05,
        0.10
    ],

    "model__max_depth": [
        2,
        3,
        4
    ],

    "model__min_child_weight": [
        1,
        3,
        5
    ],

    "model__subsample": [
        0.70,
        0.85,
        1.00
    ],

    "model__colsample_bytree": [
        0.70,
        0.85,
        1.00
    ],

    "model__gamma": [
        0,
        0.10,
        0.30
    ],

    "model__reg_alpha": [
        0,
        0.01,
        0.10
    ],

    "model__reg_lambda": [
        1,
        2,
        5
    ],

    "model__scale_pos_weight": [
        1,
        positive_class_weight
    ]
}

xgboost_search = RandomizedSearchCV(
    estimator=xgboost_search_pipeline,
    param_distributions=xgboost_parameter_space,
    n_iter=25,
    scoring=scoring_metrics,
    refit="roc_auc",
    cv=cross_validation,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    return_train_score=True
)

xgboost_search.fit(
    X_train,
    y_train
)

print("Mejores parámetros:")

for parameter, value in xgboost_search.best_params_.items():
    print(f"{parameter}: {value}")

print(
    "\nMejor ROC-AUC medio:",
    round(
        xgboost_search.best_score_,
        3
    )
)

Mejores parámetros:
model__subsample: 0.7
model__scale_pos_weight: 1
model__reg_lambda: 1
model__reg_alpha: 0.01
model__n_estimators: 200
model__min_child_weight: 1
model__max_depth: 2
model__learning_rate: 0.02
model__gamma: 0.1
model__colsample_bytree: 0.85

Mejor ROC-AUC medio: 0.843


In [23]:
xgboost_search_results = pd.DataFrame(
    xgboost_search.cv_results_
)

xgboost_top_results = (
    xgboost_search_results[
        [
            "param_model__n_estimators",
            "param_model__learning_rate",
            "param_model__max_depth",
            "param_model__min_child_weight",
            "param_model__subsample",
            "param_model__colsample_bytree",
            "param_model__gamma",
            "param_model__reg_alpha",
            "param_model__reg_lambda",
            "param_model__scale_pos_weight",
            "mean_test_accuracy",
            "mean_test_precision",
            "mean_test_sensitivity",
            "mean_test_specificity",
            "mean_test_f1",
            "mean_test_roc_auc",
            "std_test_roc_auc",
            "mean_test_pr_auc"
        ]
    ]
    .sort_values(
        "mean_test_roc_auc",
        ascending=False
    )
    .head(10)
    .reset_index(drop=True)
    .round(3)
)

xgboost_top_results

,param_model__n_estimators,param_model__learning_rate,param_model__max_depth,param_model__min_child_weight,param_model__subsample,param_model__colsample_bytree,param_model__gamma,param_model__reg_alpha,param_model__reg_lambda,param_model__scale_pos_weight,mean_test_accuracy,mean_test_precision,mean_test_sensitivity,mean_test_specificity,mean_test_f1,mean_test_roc_auc,std_test_roc_auc,mean_test_pr_auc
0,200,0.02,2,1,0.70,0.85,0.1,0.01,1,1.000,0.775,0.726,0.580,0.880,0.643,0.843,0.017,0.742
1,100,0.02,2,3,0.70,0.70,0.0,0.01,2,1.869,0.756,0.633,0.720,0.775,0.673,0.838,0.015,0.738
2,100,0.02,3,5,1.00,0.70,0.0,0.10,5,1.000,0.769,0.738,0.533,0.895,0.616,0.835,0.020,0.731
3,300,0.02,3,5,0.85,0.70,0.1,0.01,1,1.000,0.762,0.695,0.589,0.855,0.634,0.834,0.020,0.734
4,200,0.02,4,3,0.70,1.00,0.0,0.10,5,1.000,0.770,0.701,0.603,0.860,0.647,0.834,0.018,0.724
5,300,0.02,3,3,0.85,0.70,0.0,0.00,5,1.000,0.767,0.696,0.598,0.857,0.642,0.833,0.018,0.727
6,300,0.02,3,5,1.00,0.85,0.1,0.00,5,1.869,0.756,0.628,0.743,0.762,0.680,0.831,0.019,0.737
7,200,0.02,2,1,1.00,0.85,0.3,0.10,1,1.000,0.769,0.720,0.561,0.880,0.629,0.831,0.018,0.730
8,200,0.05,2,3,1.00,0.70,0.0,0.00,2,1.869,0.761,0.637,0.743,0.770,0.684,0.831,0.020,0.732
9,100,0.05,4,1,0.70,0.70,0.0,0.00,1,1.869,0.762,0.640,0.733,0.778,0.682,0.831,0.020,0.724


### Interpretación del ajuste de XGBoost

El ajuste aumentó el ROC-AUC medio de XGBoost de 0,812 a 0,843, una mejora
absoluta de 0,031.

La mejor configuración utilizó árboles poco profundos, una velocidad de
aprendizaje de 0,02 y 200 estimadores. Esta combinación construye el modelo de
forma gradual y limita la complejidad de cada árbol.

El modelo seleccionado por ROC-AUC alcanzó:

- Sensibilidad: 0,580.
- Especificidad: 0,880.
- Precisión: 0,726.
- F1: 0,643.
- ROC-AUC: 0,843.
- PR-AUC: 0,742.

La configuración seleccionada no utilizó ponderación adicional para la clase
positiva. Algunas configuraciones ponderadas obtuvieron sensibilidades superiores
a 0,72, pero presentaron menor ROC-AUC y especificidad.

Después del ajuste, XGBoost quedó muy cerca de la regresión logística. Sin
embargo, no mostró una ventaja clara que compensara su mayor complejidad y menor
interpretabilidad.

## Comparación de los modelos ajustados

Se reúnen las métricas de validación cruzada de las configuraciones seleccionadas
para cada algoritmo.

Las diferencias deben interpretarse junto con la variabilidad entre pliegues. Un
valor medio ligeramente mayor no demuestra por sí solo superioridad estadística
o clínica.

In [24]:
tuned_model_searches = {
    "Regresión logística": logistic_grid_search,
    "Random Forest": random_forest_search,
    "XGBoost": xgboost_search
}

tuned_model_results = []

for model_name, search_object in tuned_model_searches.items():

    best_index = search_object.best_index_
    cv_results = search_object.cv_results_

    result_row = {
        "modelo": model_name,

        "accuracy_media": cv_results[
            "mean_test_accuracy"
        ][best_index],

        "precision_media": cv_results[
            "mean_test_precision"
        ][best_index],

        "sensitivity_media": cv_results[
            "mean_test_sensitivity"
        ][best_index],

        "specificity_media": cv_results[
            "mean_test_specificity"
        ][best_index],

        "f1_media": cv_results[
            "mean_test_f1"
        ][best_index],

        "roc_auc_media": cv_results[
            "mean_test_roc_auc"
        ][best_index],

        "roc_auc_sd": cv_results[
            "std_test_roc_auc"
        ][best_index],

        "pr_auc_media": cv_results[
            "mean_test_pr_auc"
        ][best_index]
    }

    tuned_model_results.append(result_row)

tuned_model_results = (
    pd.DataFrame(tuned_model_results)
    .set_index("modelo")
    .sort_values(
        "roc_auc_media",
        ascending=False
    )
    .round(3)
)

tuned_model_results

,accuracy_media,precision_media,sensitivity_media,specificity_media,f1_media,roc_auc_media,roc_auc_sd,pr_auc_media
modelo,,,,,,,,
Regresión logística,0.749,0.618,0.738,0.755,0.672,0.844,0.019,0.751
XGBoost,0.775,0.726,0.580,0.880,0.643,0.843,0.017,0.742
Random Forest,0.765,0.649,0.719,0.790,0.682,0.840,0.021,0.736


## Selección del modelo

Los tres modelos ajustados obtuvieron resultados cercanos en validación cruzada:

- Regresión logística: ROC-AUC de 0,844.
- XGBoost: ROC-AUC de 0,843.
- Random Forest: ROC-AUC de 0,840.

Las diferencias son pequeñas frente a la variabilidad entre pliegues, por lo que
no existe evidencia de una superioridad marcada de un algoritmo.

Se selecciona la regresión logística como modelo final porque:

- Obtuvo el mayor ROC-AUC medio.
- Obtuvo el mayor PR-AUC.
- Presentó la mayor sensibilidad entre las configuraciones seleccionadas.
- Permite interpretar la dirección y magnitud de los coeficientes.
- Requiere menor complejidad computacional.
- Su desempeño fue igual o superior al de los modelos basados en árboles.

Random Forest presentó el mayor F1 y una mejor especificidad, por lo que
constituye una alternativa razonable. Sin embargo, su ventaja no fue suficiente
para desplazar a la regresión logística como modelo principal.

La selección se realizó exclusivamente con datos de entrenamiento. El conjunto
de prueba todavía no ha sido utilizado.